# Collaborative factorization (architecture B) — results

**Compute runs externally**; this notebook only loads the results and plots them.

Isolates the **y-ceiling**: factorize ONLY the train interaction matrix (no embeddings),
`A_signed = S_pos - S_neg ≈ U Vᵀ` (truncated SVD), giving a molecule factor `u_m` and a receptor
factor `v_p` learned purely from who-binds-what. Then measure how much this pure-`y` representation
adds on top of — or instead of — the raw embeddings the boost already reads. Bars:

- **raw** — `[mol | prot]`, x only.
- **mf_prot** — + `v_p` (receptor collaborative factor).
- **mf_mol** — + `u_m` (molecule collaborative factor).
- **mf_both** — + `u_m + v_p`.
- **mf_only** — `[u_m | v_p]`, **no embeddings** — the pure-`y` signal in isolation.

**Cold-start is the point:** in `inductive_molecule` the held-out molecule has no train edges → its
row is zero → `u_m ≈ 0`, so `mf_mol / mf_both / mf_only` collapse toward protein-only. That visible
collapse is the result — pure-`y` molecule factors can't reach a cold molecule (why the graph needs
node features). In transductive both factors are live.

Both regimes, **5 repeats** each. Inductive cold split **stratified to matched prevalence**.
Embeddings standardised; MF factors keep their SVD scale (boost is scale-invariant).

### Run the compute
```bash
bash scripts/modeling/eval/run_collaborative_factorization.sh            # live dashboard
MAX_PARALLEL=10 bash scripts/modeling/eval/run_collaborative_factorization.sh
```
Then run the cells below.

In [ ]:
import sys, pathlib, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import t as student_t

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('run from inside the orbind repository')
    ROOT = ROOT.parent

RESULT_DIR = ROOT / 'results/graph/collab_factorization'
COMBINED = RESULT_DIR / 'tables/collab_factorization_runs.csv'
RUNS_DIR = RESULT_DIR / 'tables/runs'
FIGURES_DIR = ROOT / 'notebooks/figures'; FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS = ['AUROC', 'AUPRC', 'MCC', 'F1']
REGIMES = ['transductive', 'inductive_molecule']

# prefer the combined CSV; fall back to globbing per-unit CSVs (partial runs still render)
if COMBINED.exists():
    RES = pd.read_csv(COMBINED)
    print('loaded', COMBINED.relative_to(ROOT))
elif RUNS_DIR.exists() and glob.glob(str(RUNS_DIR / '*.csv')):
    RES = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(RUNS_DIR / '*.csv')))], ignore_index=True)
    print(f'loaded {RES.repeat.nunique()} repeat-files from', RUNS_DIR.relative_to(ROOT))
else:
    RES = pd.DataFrame()
    print('No results yet. Run:  bash scripts/modeling/eval/run_collaborative_factorization.sh')

def ci95_half(x):
    x = pd.Series(x).dropna().astype(float)
    if len(x) < 2:
        return np.nan
    return float(student_t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))

if len(RES):
    n = RES.groupby('regime').repeat.nunique().to_dict()
    print('repeats per regime:', n)
    print('mean test prevalence:', RES.groupby('regime').test_prevalence.mean().round(3).to_dict())
    # SVD diagnostics: energy captured by the rank, and the cold-start collapse fraction
    diag = RES.groupby('regime')[['mf_rank', 'sv_energy', 'zero_factor_mols']].mean().round(3)
    print('\nSVD diagnostics (rank; singular-value energy captured; fraction of zero molecule-factors):')
    display(diag)
    display(RES.groupby(['regime', 'bar'])[METRICS].mean().round(3))

In [ ]:
# ---- bar-grid: regimes (rows) x metrics (cols), mean +/- 95% t-CI over repeats ----
ORDER = ['raw', 'mf_prot', 'mf_mol', 'mf_both', 'mf_only']
LABEL = {'raw': 'raw', 'mf_prot': '+v_p', 'mf_mol': '+u_m', 'mf_both': '+u+v', 'mf_only': 'u,v\nonly'}
# grey baseline; MF-augmented in blues; the pure-y bar in purple
BAR_C = {'raw': '#9a9a95', 'mf_prot': '#8fb0e6', 'mf_mol': '#5a8fd6', 'mf_both': '#2a78d6',
         'mf_only': '#8a5cc0'}

if not len(RES):
    print('No results to plot yet.')
else:
    fig, axes = plt.subplots(len(REGIMES), len(METRICS), figsize=(15, 7), squeeze=False)
    for ri, regime in enumerate(REGIMES):
        d = RES[RES.regime == regime]
        bars = [b for b in ORDER if b in set(d.bar)]
        for mi, metric in enumerate(METRICS):
            ax = axes[ri, mi]
            means = [d[d.bar == b][metric].mean() for b in bars]
            errs = [ci95_half(d[d.bar == b][metric]) for b in bars]
            x = np.arange(len(bars))
            ax.bar(x, means, color=[BAR_C[b] for b in bars], edgecolor='white', linewidth=1.2,
                   yerr=np.nan_to_num(errs), error_kw=dict(ecolor='#33322f', capsize=3, lw=1), zorder=3)
            if means:
                ax.axhline(means[0], color=BAR_C['raw'], ls='--', lw=1, zorder=1)   # raw baseline
            ax.set_ylim(0, 1); ax.grid(alpha=.15, axis='y')
            ax.set_xticks(x); ax.set_xticklabels([LABEL[b] for b in bars], fontsize=8)
            for s in ('top', 'right'): ax.spines[s].set_visible(False)
            if ri == 0: ax.set_title(metric, fontsize=11)
            for xi, v in zip(x, means):
                if np.isfinite(v):
                    ax.text(xi, v + .02, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
        nrep = d.repeat.nunique()
        axes[ri, 0].set_ylabel(f'{regime}\n({nrep} repeats)', fontsize=10)
    fig.suptitle('Collaborative factorization (pure-y latent) — mean +/- 95% t-CI '
                 '(stratified inductive split; purple = no embeddings)', fontsize=12, y=1.0)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'collaborative_factorization.png', dpi=150, bbox_inches='tight')
    plt.show()

## How to read

- **mf_only (purple)** is the y-ceiling in isolation — prediction from collaborative factors with **no
  embeddings**. High in **transductive** ⇒ the interaction matrix alone carries real signal; near-random
  in **inductive** ⇒ pure-`y` cannot reach a cold molecule (cold-start wall). The `zero_factor_mols`
  diagnostic quantifies that collapse.
- **The additivity question is mf_both vs raw** (transductive): if the pure-`y` latent adds nothing on top
  of the embeddings the boost already reads, `y` is redundant with `x` for warm molecules — the
  transductive-ceiling story.
- **mf_prot** is expected ≈ raw: the receptor factor `v_p` is essentially receptor identity, which the
  boost already reads from `x_p`.
- Compare on **AUROC** primarily (prevalence-independent; stratified split matches val/test prevalence).
  Whiskers are 95% t-CI over 5 repeats. `sv_energy` shows how much of the matrix the rank captures.

Knobs via env: `RANK` (latent dimension), `MAX_PARALLEL`. Delete
`results/graph/collab_factorization/tables/` to regenerate.